# 3.5 Imbalanced Classification

使用 class weight 處理不平衡二元分類。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve, auc

## 1. 建立不平衡資料

In [ ]:
X, y = make_classification(
    n_samples=12000,
    n_features=20,
    n_informative=8,
    n_redundant=4,
    weights=[0.95, 0.05],
    flip_y=0.01,
    random_state=42
)

pd.Series(y).value_counts(normalize=True).rename('ratio')

## 2. 切分與標準化

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

## 3. 設定 class weight

In [ ]:
classes = np.unique(y_train)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weight = dict(zip(classes, weights))
class_weight

## 4. 建立模型

In [ ]:
tf.keras.utils.set_random_seed(42)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(x_train.shape[1],)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc'), tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')]
)

## 5. 訓練模型

In [ ]:
history = model.fit(
    x_train, y_train,
    validation_split=0.2,
    epochs=40,
    batch_size=128,
    class_weight=class_weight,
    verbose=0
)

pd.DataFrame(history.history)[['loss', 'val_loss']].plot(figsize=(8, 4), title='Loss Curve')
plt.show()

## 6. 評估模型

In [ ]:
result = model.evaluate(x_test, y_test, verbose=0, return_dict=True)
pd.DataFrame({'metric': list(result.keys()), 'test': list(result.values())})

In [ ]:
y_prob = model.predict(x_test, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

## 7. 調整 threshold

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_test, y_prob)
pr_auc = auc(recall, precision)

plt.figure(figsize=(6, 4))
plt.plot(recall, precision, label=f'PR AUC={pr_auc:.3f}')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.legend()
plt.grid(True)
plt.show()

for threshold in [0.3, 0.5, 0.7]:
    pred = (y_prob >= threshold).astype(int)
    print(f'\nThreshold = {threshold}')
    print(confusion_matrix(y_test, pred))